In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate

import ADFWI
from ADFWI.model import AbstractModel, AcousticModel, AnisotropicElasticModel, IsotropicElasticModel
from ADFWI.propagator import AcousticPropagator, ElasticPropagator, GradProcessor, TorchGradProcessor
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import build_anomaly_background_model, build_layer_model, get_anomaly_model, get_linear_hess_model, get_linear_marmousi2_model, get_linear_vel_model, get_smooth_hess_model, get_smooth_layer_model, get_smooth_marmousi_model, get_smooth_valhall_model, load_hess_model, load_marmousi_model, load_overthrust_initial_model, load_overthrust_model, load_valhall_model, numpy2tensor, resample_marmousi_model, resample_overthrust_model, tensor2numpy, wavelet
from ADFWI.view import animate_inversion_process, plot_bcx_bcz, plot_damp, plot_eps_delta_gamma, plot_initial_and_inverted, plot_lam_mu, plot_misfit, plot_model, plot_survey, plot_vp_rho, plot_vp_vs_rho, plot_waveform2D, plot_waveform_trace, plot_waveform_wiggle, plot_wavelet


## Basic Parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32
backend = ADFWI.set_backend(device, dtype=dtype)
ox,oz = 0,0
nz,nx = 100,100
dx,dz = 10,10
nt,dt = 2501,0.0002
nabc  = 50
f0    = 15
free_surface = False

## Velocity Model

In [ ]:
# velocity model 
vp                      = np.ones((nz,nx))
vp[:,:]               = 3724

vs                      = np.ones((nz,nx))
vs[:,:]               = 1944

rho                     = np.ones((nz,nx))
rho[:,:]              = 2450

model = IsotropicElasticModel(ox,oz,nx,nz,dx,dz,
                              vp,vs,rho,
                              vp_grad=False,
                              vs_grad=False,
                              auto_update_rho=False,auto_update_vp=False,
                              free_surface=free_surface,
                              abc_type="PML",abc_jerjan_alpha=0.007,
                              nabc=nabc
                    )
print(model.__repr__())
# model.save("./examples/waveform_checking/HomogeousModel-Isotropic/data/model.npz")

In [ ]:
model._plot_vp_vs_rho(figsize=(12,5),wspace=0.3,cbar_pad_fraction=0.18,cmap='jet_r')
model._plot_lam_mu(figsize=(12,5),wspace=0.15,cbar_pad_fraction=0.18,cmap='jet_r')

GRTM
## Source and Receiver

In [ ]:
src_z = np.array([50]) 
src_x = np.array([99])
src_t,src_v = wavelet(nt,dt,f0,amp0=1)
# src_v = -integrate.cumtrapz(src_v, axis=-1, initial=0) #Integrate
src_v = -src_v
source = Source(nt=nt,dt=dt,f0=f0)

for i in range(len(src_x)):
    source.add_source(src_x=src_x[i],src_z=src_z[i],src_wavelet=src_v,src_type="mt",src_mt=np.array([[0,0,1],[0,0,0],[0,0,0]]))

source.plot_wavelet()

In [ ]:
if free_surface:
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/free_surface_version2/wavelet.txt",src_v)
else:
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/wavelet.txt",src_v)

In [ ]:
# receiver
rcv_z = np.array([1  for i in range(0,nx,1)])
rcv_x = np.array([j for j in range(0,nx,1)])
receiver = Receiver(nt=nt,dt=dt)
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i],rcv_z=rcv_z[i],rcv_type="pr")

survey = Survey(source=source,receiver=receiver)
print(survey.__repr__())

In [ ]:
survey.plot(model.vs,cmap='jet_r')

In [ ]:
import json
if free_surface:
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/free_surface_version2/vp.txt",vp)
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/free_surface_version2/vs.txt",vs)
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/free_surface_version2/rho.txt",rho)
else:
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/vp.txt",vp)
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/vs.txt",vs)
    np.savetxt("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/rho.txt",rho)
params = {
    "dx":dx,
    "dz":dz,
    "nx":nx,
    "nz":nz,
    "nt":nt,
    "dt":dt,
    "pml":nabc,
    "f0":f0,
    "src":(np.hstack((src_x.reshape(-1,1),src_z.reshape(-1,1)))*dx).tolist(),
    "rcv":(np.hstack((rcv_x.reshape(-1,1),rcv_z.reshape(-1,1)))*dx).tolist(),
}
json_str = json.dumps(params)
if free_surface:
    with open("./examples/waveform_checking/HomogeousModel-Isotropic/data/free_surface_version2/params.json",'w') as json_file:
        json_file.write(json_str)
else:
    with open("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/params.json",'w') as json_file:
        json_file.write(json_str)

## Propagator

In [ ]:
F = ElasticPropagator(model,survey)

In [ ]:
if model.abc_type == "PML":
    bcx = F.bcx
    bcz = F.bcz
    title_param = {'family':'Times New Roman','weight':'normal','size': 15}
    plot_bcx_bcz(bcx,bcz,dx=dx,dz=dz,wspace=0.25,title_param=title_param,cbar_height=0.04,cbar_pad_fraction=0.12)
else:
    damp = F.damp
    plot_damp(damp)

In [ ]:
record_waveform = F.forward(fd_order=4)
rcv_txx,rcv_tzz,rcv_txz,rcv_vx,rcv_vz = record_waveform["txx"],record_waveform["tzz"],record_waveform["txz"],record_waveform["vx"],record_waveform["vz"]
forward_wavefield_txx,forward_wavefield_tzz,forward_wavefield_txz,forward_wavefield_vx,forward_wavefield_vz = record_waveform["forward_wavefield_txx"],record_waveform["forward_wavefield_tzz"],record_waveform["forward_wavefield_txz"],record_waveform["forward_wavefield_vx"],record_waveform["forward_wavefield_vz"]

In [ ]:
d_obs = SeismicData(survey)
d_obs.record_data(record_waveform)
if free_surface:
    d_obs.save("./examples/waveform_checking/HomogeousModel-Isotropic/data/free_surface_version2/obs_data.npz")
else:
    d_obs.save("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/obs_data.npz")

## Plotting Figure

In [ ]:
# waveform
shot = 0
plot_waveform2D(-(rcv_txx+rcv_tzz)[shot].T,figsize=(6,6),cmap='coolwarm')
plot_waveform_wiggle(-(rcv_txx+rcv_tzz)[shot],source.t,show=True)

In [ ]:
plt.figure()
plt.pcolormesh(-(rcv_tzz+rcv_txx)[0].cpu().detach().numpy())
plt.gca().invert_yaxis()
plt.show()

In [ ]:
shot = 0
trace = -1
plot_waveform_trace(-(rcv_tzz+rcv_txx),shot,trace,dt=dt)

In [ ]:
shot = 0
trace = -1
plot_waveform_trace(rcv_vx,shot,trace,dt=dt)

In [ ]:
import obspy
from obspy.io.segy.core import _read_segy

specfem_vx_st = obspy.read("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/Ux_file_single_v000000.su",format="SU",byteorder="<") 
specfem_vz_st = obspy.read("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/Uz_file_single_v000000.su",format="SU",byteorder="<") 
specfem_p_st  = obspy.read("./examples/waveform_checking/HomogeousModel-Isotropic/data/no_free_surface_version2/Up_file_single_p000000.su",format="SU",byteorder="<") 

ADFWI_rcv_p = -(rcv_tzz+rcv_txx)

def normalize(data):
    return (data - data.mean())/(data.max() - data.min())

trace = 0

plt.figure(figsize=(12,4))
tlist = np.arange(len(specfem_vz_st[0].data))*0.0002
plt.plot(tlist+0.002,normalize(-specfem_p_st[trace].data),c='r',linestyle='-',label='specfem')
plt.plot(tlist,normalize(ADFWI_rcv_p[0,:,trace].cpu().detach().numpy()),c='k',linestyle="--",label='ADFWI')
# plt.xlim(0.4)
# plt.ylim(-0.1,0.1)
plt.legend()
plt.show()

In [ ]:
ADFWI_rcv_vz = rcv_vz
plt.figure()
trace = 0
plt.plot(tlist+0.002,normalize(specfem_vz_st[trace].data),c='r',linestyle='-',label='specfem')
plt.plot(tlist,normalize(ADFWI_rcv_vz[0,:,trace].cpu().detach().numpy()),c='k',linestyle="--",label='ADFWI')
# plt.xlim(0.3)
# plt.ylim(-0.1,0.1)
plt.legend()
plt.show()

In [ ]:
ADFWI_rcv_vx = rcv_vx
plt.figure()
trace = 0

plt.plot(tlist+0.002,normalize(-specfem_vx_st[trace].data),c='r',linestyle='-',label='specfem')
plt.plot(tlist,normalize(ADFWI_rcv_vx[0,:,trace].cpu().detach().numpy()),c='k',linestyle="--",label='ADFWI')
plt.legend()
# plt.xlim(0.4,0.5)
# plt.ylim(-0.1,0.1)
plt.show()

## devito

In [ ]:
devito_acoustic_p = np.load("/media/liufeng/a0b205ec-bfb3-473f-a6f0-0680c5da64ba/project/004_inversion/ADInversion/Others_work/Devito/devito/example_LF/data/elastic/homo/shot0.npz")["data"]

In [ ]:
shot = 0
trace = 99

plt.figure()
plt.plot(normalize(ADFWI_rcv_p[shot,:,trace]),c='k',linestyle="--",label="ADFWI")
plt.plot(normalize(-devito_acoustic_p[105:,trace]),c='r',linestyle="--",label="Devito")
plt.legend()
plt.show()